# LangGraph Coordination Layer — Basic Search (baseline)

Three-node `StateGraph` implementing Section 6.3, with **Basic Search** as the retrieval mode (baseline RAG condition):

```
get_classification_context  →  classify_ticket  →  resolve_ticket  →  END
```

Identical workflow to the DRIFT version — only the retrieval call inside the content nodes differs. Internally this calls `graphrag.api.basic_search`, which is the same code path that `graphrag query --method basic` dispatches to.

Per question, writes 4 artefacts into `answers_langgraph_basic/<question_stem>/`:

- `classification_answer.txt` — raw text from node 1
- `resolution_answer.txt` — raw text from node 3
- `combined.json` — appendix log (ticket + both answers + structured classification)
- `coordination_log.json` — execution trace + criteria evidence (Section 7.3.3)

Run from the GraphRAG project root (the folder with `settings.yaml` and `output/`).


In [2]:
import httpx
import json

_original_async_send = httpx.AsyncClient.send
_original_sync_send  = httpx.Client.send

def _log_payload(request):
    url = str(request.url)
    if "openai.com" not in url and "openai.azure.com" not in url:
        return
    try:
        body = json.loads(request.content.decode())
    except Exception:
        print(f"[openai] {request.method} {request.url.path} (body not JSON)")
        return
    relevant = {k: body[k] for k in ("model", "temperature", "seed", "top_p", "n") if k in body}
    print(f"[openai] {request.method} {request.url.path} -> {relevant}")

async def _async_send_with_log(self, request, **kwargs):
    _log_payload(request)
    return await _original_async_send(self, request, **kwargs)

def _sync_send_with_log(self, request, **kwargs):
    _log_payload(request)
    return _original_sync_send(self, request, **kwargs)

httpx.AsyncClient.send = _async_send_with_log
httpx.Client.send      = _sync_send_with_log

print("OpenAI request logging enabled.")

OpenAI request logging enabled.


## Imports

In [3]:
import os
import json
import time
import asyncio
import uuid
from pathlib import Path
from datetime import datetime
from typing import TypedDict, Literal
try:
    from typing import NotRequired
except ImportError:
    from typing_extensions import NotRequired  # Python 3.10 fallback

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

from graphrag.config.load_config import load_config
from graphrag.api import basic_search

load_dotenv()
print("Imports OK")


/Users/kompzen/Offline/LLM-Graphrag/GraphRAG-chunksize/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## Configuration

`ROOT` is resolved from `Path.cwd()`. The cwd must be your GraphRAG project root (with `settings.yaml` and `output/`).

In [4]:
ROOT          = Path.cwd()
QUESTIONS_DIR = ROOT / "QUERY_GLOBAL"
ANSWERS_DIR   = ROOT / "answers_langgraph_basic"

EXPECTED_NODE_SEQUENCE = [
    "get_classification_context",
    "classify_ticket",
    "resolve_ticket",
]

llm = ChatOpenAI(
    api_key=os.environ["GRAPHRAG_API_KEY"],
    model="gpt-5-mini",
    temperature=0,
)

print(f"ROOT          = {ROOT}")
print(f"QUESTIONS_DIR = {QUESTIONS_DIR}")
print(f"ANSWERS_DIR   = {ANSWERS_DIR}")


ROOT          = /Users/kompzen/Offline/LLM-Graphrag/GraphRAG-chunksize
QUESTIONS_DIR = /Users/kompzen/Offline/LLM-Graphrag/GraphRAG-chunksize/QUERY_GLOBAL
ANSWERS_DIR   = /Users/kompzen/Offline/LLM-Graphrag/GraphRAG-chunksize/answers_langgraph_basic


## Basic Search context

Basic Search only consults `text_units.parquet` (and its embeddings in the vector store). The config and the text-units table are loaded once per session and reused across all queries. Re-run this cell if you change the GraphRAG config or rebuild the index.

In [5]:
_config = None
_text_units = None

def _get_config():
    global _config
    if _config is None:
        print(f"[init] Loading GraphRAG config from {ROOT}")
        _config = load_config(ROOT)
    return _config

def _get_text_units():
    global _text_units
    if _text_units is None:
        out_dir = ROOT / "output"
        text_units_path = out_dir / "text_units.parquet"
        if not text_units_path.exists():
            raise FileNotFoundError(f"Missing: {text_units_path.resolve()}")
        _text_units = pd.read_parquet(text_units_path)
        print(f"[init] Loaded {len(_text_units)} text units")
    return _text_units

print("Basic Search context loader ready (lazy on first query)")


Basic Search context loader ready (lazy on first query)


## Basic Search helper

Wraps `graphrag.api.basic_search` — the same code path the `graphrag query --method basic` CLI dispatches to, called in-process. Returns the response text only; the retrieved chunks are discarded since Basic Search performs a single k-NN lookup and there are no per-step decisions to record.

In [6]:
async def graphrag_basic(query: str) -> str:
    """Run Basic Search and return the response text."""
    response, _context = await basic_search(
        config=_get_config(),
        text_units=_get_text_units(),
        query=query,
    )
    return response


In [ ]:
! pip show graphrag

## Stage prompts

Constants combined with the ticket text at runtime. Same prompts as the DRIFT condition so prompt design does not act as a confound between the two retrieval modes (Section 7.1.3). Edit and re-run this cell to change the prompts — the next workflow run will pick them up immediately.

In [7]:
CLASSIFICATION_PROMPT = """\
You are reviewing an IT support ticket. Based on similar past cases in the
indexed knowledge base, classify this ticket as either first_level_support
or second_level_support.

- first_level_support: routine issues that were resolved without engineer
  involvement in similar past cases (standard procedures, documented fixes).
- second_level_support: complex issues that required engineer involvement
  or escalation in similar past cases.

Recommend a classification, briefly explain the reasoning by referring to
similar past cases, and cite the source documents you relied on.

--- Ticket ---
{ticket}
"""

RESOLUTION_PROMPT = """\
You are resolving an IT support ticket. Using the indexed organisational
knowledge — known fixes, documented procedures, and prior resolutions —
provide a clear, step-by-step resolution for the ticket below.

If the indexed knowledge does not contain enough information to fully
resolve the ticket, say so explicitly. Cite the source documents for
each step.

--- Ticket ---
{ticket}
"""


## Classification structuring

Schema + chain used by node 2 to convert GraphRAG's recommendation into a typed value. The LLM here does NOT classify — it only structures the recommendation GraphRAG already produced.

In [8]:
class TicketClassification(BaseModel):
    classification: Literal["first_level_support", "second_level_support"] = Field(
        description="Classification recommended by the GraphRAG response."
    )
    reason: str = Field(
        description="Short explanation including source citations from the GraphRAG response."
    )

structuring_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You convert a GraphRAG response into a typed classification result. "
     "You do NOT classify the ticket yourself — GraphRAG has already done so. "
     "Your job is only to extract the recommended classification and a short "
     "cited reason from the response provided."),
    ("human",
     "Ticket:\n{ticket}\n\nGraphRAG response:\n{context}"),
])

structuring_chain = structuring_prompt | llm.with_structured_output(TicketClassification)


## Shared state + nodes

Edit any node and re-run this cell + the next one (compile graph) to pick up the changes.

In [9]:
class AgentState(TypedDict):
    ticket_description: str
    classification_context: NotRequired[str]
    classification_output:  NotRequired[TicketClassification]
    resolution_output:      NotRequired[str]


async def get_classification_context(state: AgentState) -> dict:
    print("--- Node 1: get_classification_context (Basic Search) ---")
    query = CLASSIFICATION_PROMPT.format(ticket=state["ticket_description"])
    response = await graphrag_basic(query)
    return {"classification_context": response}

async def classify_ticket(state: AgentState) -> dict:
    print("--- Node 2: classify_ticket (structuring LLM) ---")
    result = await structuring_chain.ainvoke({
        "ticket": state["ticket_description"],
        "context": state["classification_context"],
    })
    return {"classification_output": result}

async def resolve_ticket(state: AgentState) -> dict:
    """Classification result is preserved in state but NOT consumed here
    (Section 6.3.3, second design choice)."""
    print("--- Node 3: resolve_ticket (Basic Search, no structuring) ---")
    query = RESOLUTION_PROMPT.format(ticket=state["ticket_description"])
    response = await graphrag_basic(query)
    return {"resolution_output": response}


## Compile graph

In [10]:
graph = StateGraph(AgentState)
graph.add_node("get_classification_context", get_classification_context)
graph.add_node("classify_ticket",            classify_ticket)
graph.add_node("resolve_ticket",             resolve_ticket)

graph.set_entry_point("get_classification_context")
graph.add_edge("get_classification_context", "classify_ticket")
graph.add_edge("classify_ticket",            "resolve_ticket")
graph.add_edge("resolve_ticket",             END)

rag_agent = graph.compile(checkpointer=MemorySaver())
print("Graph compiled")


Graph compiled


## Coordination log builder

Reads LangGraph's checkpoint history after a run and produces structured evidence for the three coordination criteria from Section 7.3.3:

- **Memory** — were upstream outputs still in state when `resolve_ticket` ran?
- **Execution and actuation** — was GraphRAG invoked separately per stage with task-specific prompts?
- **Orchestration and workflow control** — did nodes execute in the expected order and terminate after `resolve_ticket`?


In [11]:
# Mapping from "which node just executed" to "which state keys appear AFTER that node"
NODE_OUTPUT_KEYS = {
    "get_classification_context": {"classification_context"},
    "classify_ticket":            {"classification_output"},
    "resolve_ticket":             {"resolution_output"},
}

# Which nodes invoke GraphRAG (vs the structuring-only node).
GRAPHRAG_NODES = {
    "get_classification_context": "classification",
    "resolve_ticket":             "resolution",
}


def _infer_node_from_diff(prev_keys: set, curr_keys: set) -> str | None:
    """Figure out which node executed by looking at which state keys appeared.

    More robust than reading metadata['writes'] — works across LangGraph versions.
    """
    new_keys = curr_keys - prev_keys
    if not new_keys:
        return None
    for node, expected in NODE_OUTPUT_KEYS.items():
        if expected.issubset(new_keys):
            return node
    return None


def build_coordination_log(agent, thread_config, question_file):
    snapshots = list(agent.get_state_history(thread_config))
    snapshots.reverse()  # chronological

    steps = []
    observed_sequence = []
    prev_keys: set = set()

    for snap in snapshots:
        meta = snap.metadata or {}
        values = snap.values or {}
        curr_keys = set(values.keys())

        node = _infer_node_from_diff(prev_keys, curr_keys)
        wrote_keys = sorted(curr_keys - prev_keys)

        if node and node not in observed_sequence:
            observed_sequence.append(node)

        # GraphRAG was invoked iff one of the GraphRAG-invoking nodes just
        # executed. The prompt is task-specific by construction (each node
        # uses a distinct hard-coded prompt template).
        graphrag_invoked = node in GRAPHRAG_NODES
        prompt_kind      = GRAPHRAG_NODES.get(node)

        next_nodes = list(snap.next) if snap.next else []

        steps.append({
            "step": meta.get("step"),
            "timestamp": getattr(snap, "created_at", None),
            "source": meta.get("source"),
            "node_just_executed": node,
            "wrote_keys": wrote_keys,
            "state_keys_present": sorted(curr_keys),
            "graphrag_invoked": graphrag_invoked,
            "graphrag_prompt_kind": prompt_kind,
            "next_nodes": next_nodes,
        })

        prev_keys = curr_keys

    # Memory: when resolve_ticket is about to run (i.e. it's the next node),
    # are the upstream outputs already in state?
    pre_resolve_step = next(
        (s for s in steps if "resolve_ticket" in s["next_nodes"]),
        None,
    )
    cls_ctx_present = bool(
        pre_resolve_step and "classification_context" in pre_resolve_step["state_keys_present"]
    )
    cls_out_present = bool(
        pre_resolve_step and "classification_output" in pre_resolve_step["state_keys_present"]
    )

    invocations = [
        {"node": s["node_just_executed"], "prompt_kind": s["graphrag_prompt_kind"]}
        for s in steps if s["graphrag_invoked"]
    ]
    invocations_per_stage = {}
    for inv in invocations:
        invocations_per_stage[inv["prompt_kind"]] = invocations_per_stage.get(inv["prompt_kind"], 0) + 1
    task_specific = "classification" in invocations_per_stage and "resolution" in invocations_per_stage

    sequence_matches = observed_sequence == EXPECTED_NODE_SEQUENCE
    last = steps[-1] if steps else None
    terminated = bool(
        last
        and last["node_just_executed"] == "resolve_ticket"
        and not last["next_nodes"]
    )

    return {
        "question_file": question_file,
        "thread_id": thread_config["configurable"]["thread_id"],
        "retrieval_mode": "basic",
        "expected_node_sequence": EXPECTED_NODE_SEQUENCE,
        "observed_node_sequence": observed_sequence,
        "steps": steps,
        "criteria_evidence": {
            "memory": {
                "classification_context_present_at_resolve_ticket": cls_ctx_present,
                "classification_output_present_at_resolve_ticket": cls_out_present,
            },
            "execution_and_actuation": {
                "graphrag_invocations": invocations,
                "invocations_per_stage": invocations_per_stage,
                "task_specific_prompts": task_specific,
            },
            "orchestration_and_workflow_control": {
                "node_sequence_matches_expected": sequence_matches,
                "terminated_after_resolve_ticket": terminated,
            },
        },
    }


## Artefact writers + `run_one`

`run_one` returns `(qdir, state, coord_log)` so you can inspect a single test run inline.

In [12]:
def _write_text(path, text):
    path.write_text(text, encoding="utf-8")

def _write_json(path, payload):
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def _write_artefacts(qdir, ticket, ticket_file, state, coord_log):
    cls = state["classification_output"]

    _write_text(qdir / "classification_answer.txt", state["classification_context"])
    _write_text(qdir / "resolution_answer.txt",     state["resolution_output"])

    _write_json(qdir / "combined.json", {
        "question_file": ticket_file,
        "ticket": ticket,
        "retrieval_mode": "basic",
        "classification_answer":     state["classification_context"],
        "classification_structured": cls.model_dump(),
        "resolution_answer":         state["resolution_output"],
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    })

    _write_json(qdir / "coordination_log.json", coord_log)

async def run_one(question_path: Path, answers_root: Path = ANSWERS_DIR):
    """Run a single question and write all 4 artefacts. Returns (qdir, state, coord_log)."""
    qdir = answers_root / question_path.stem
    qdir.mkdir(parents=True, exist_ok=True)

    ticket = question_path.read_text(encoding="utf-8").strip()
    thread = {"configurable": {"thread_id": str(uuid.uuid4())}}

    state = await rag_agent.ainvoke({"ticket_description": ticket}, thread)
    coord_log = build_coordination_log(rag_agent, thread, question_path.name)

    _write_artefacts(qdir, ticket, question_path.name, state, coord_log)
    return qdir, state, coord_log


## Test on a single query

Use this to verify the workflow end-to-end before kicking off the full batch. Edit `TEST_QUESTION` to point at whichever ticket you want to try.

In [13]:
TEST_QUESTION = QUESTIONS_DIR / "query8.txt"

qdir, state, coord_log = await run_one(TEST_QUESTION)

print(f"\n✓ artefacts written to: {qdir}")
print(f"\nClassification: {state['classification_output'].classification}")
print(f"Reason: {state['classification_output'].reason[:200]}...")

print(f"\n--- Resolution preview (first 500 chars) ---")
print(state['resolution_output'][:500] + "...")

print(f"\n--- Coordination criteria ---")
print(json.dumps(coord_log['criteria_evidence'], indent=2))


--- Node 1: get_classification_context (Basic Search) ---
[init] Loading GraphRAG config from /Users/kompzen/Offline/LLM-Graphrag/GraphRAG-chunksize
[init] Loaded 43 text units
[openai] POST /v1/embeddings -> {'model': 'text-embedding-3-small'}
[openai] POST /v1/chat/completions -> {'model': 'gpt-5-mini', 'n': 1}
--- Node 2: classify_ticket (structuring LLM) ---
[openai] POST /v1/chat/completions -> {'model': 'gpt-5-mini'}
--- Node 3: resolve_ticket (Basic Search, no structuring) ---
[openai] POST /v1/embeddings -> {'model': 'text-embedding-3-small'}
[openai] POST /v1/chat/completions -> {'model': 'gpt-5-mini', 'n': 1}

✓ artefacts written to: /Users/kompzen/Offline/LLM-Graphrag/GraphRAG-chunksize/answers_langgraph_basic/query8

Classification: second_level_support
Reason: GraphRAG recommends second_level_support: D365 Financial Reports matches KF11/TI-722511 requiring Data Mart/FRDB work and MS support [Sources (0,1)]; new hotline numbers map to SBC routing/translation...

--- Resolut

## Batch runner

`run_batch` walks every `*.txt` in `QUERY_GLOBAL/` and runs the workflow on each one. Skips questions whose `combined.json` already exists. Failures don't write `combined.json`, so failed questions auto-retry on the next run.

`batch_status` reports what's done and what's pending without doing any work — handy for checking progress before/after a batch.

In [ ]:
def _is_done(q_file: Path, answers_root: Path = ANSWERS_DIR) -> bool:
    """A question is 'done' if its combined.json marker exists and is non-empty."""
    marker = answers_root / q_file.stem / "combined.json"
    return marker.exists() and marker.stat().st_size > 0


def batch_status(questions_dir: Path = QUESTIONS_DIR,
                 answers_root:  Path = ANSWERS_DIR,
                 show_next: int = 10) -> dict:
    """Show progress without running anything."""
    questions = sorted(questions_dir.glob("*.txt"))
    if not questions:
        print(f"No questions in {questions_dir}")
        return {"done": [], "pending": []}

    done    = [q.name for q in questions if _is_done(q, answers_root)]
    pending = [q.name for q in questions if not _is_done(q, answers_root)]

    print(f"Total questions in {questions_dir.name}/: {len(questions)}")
    print(f"  ✓ Done:     {len(done)}")
    print(f"  ◯ Pending:  {len(pending)}")

    if pending:
        print(f"\nNext up:")
        for p in pending[:show_next]:
            print(f"  ◯ {p}")
        if len(pending) > show_next:
            print(f"  ... and {len(pending) - show_next} more")

    return {"done": done, "pending": pending}


async def run_batch(questions_dir: Path = QUESTIONS_DIR,
                    answers_root:  Path = ANSWERS_DIR,
                    overwrite: bool = False) -> dict:
    if not questions_dir.exists():
        raise FileNotFoundError(f"Questions folder not found: {questions_dir}")
    answers_root.mkdir(parents=True, exist_ok=True)

    question_files = sorted(questions_dir.glob("*.txt"))
    if not question_files:
        print(f"No .txt questions found in {questions_dir}")
        return {"processed": [], "skipped": [], "failed": []}

    if overwrite:
        to_run, skipped_upfront = question_files, []
    else:
        to_run         = [q for q in question_files if not _is_done(q, answers_root)]
        skipped_upfront = [q for q in question_files if _is_done(q, answers_root)]

    total     = len(question_files)
    n_to_run  = len(to_run)
    n_skipped = len(skipped_upfront)

    print(f"Found {total} question(s) in {questions_dir.name}/")
    print(f"  Already done: {n_skipped}")
    print(f"  To run:       {n_to_run}")

    if n_to_run == 0:
        print("\nNothing to do — all questions already have answers.")
        return {"processed": [], "skipped": [q.name for q in skipped_upfront], "failed": []}

    print(f"\nStarting at {datetime.now().strftime('%H:%M:%S')}")
    batch_t0 = time.time()
    processed, failed = [], []

    for i, q_file in enumerate(to_run, start=1):
        upcoming = to_run[i].name if i < n_to_run else "— (last one)"
        print(f"\n[{i}/{n_to_run}] running: {q_file.name}   (next: {upcoming})")

        t0 = time.time()
        try:
            written, _, _ = await run_one(q_file, answers_root)
            elapsed = time.time() - t0
            print(f"[{i}/{n_to_run}] ✓ done   {q_file.name} in {elapsed:.1f}s  →  {written.name}/")
            processed.append(q_file.name)
        except Exception as e:
            elapsed = time.time() - t0
            print(f"[{i}/{n_to_run}] ✗ failed {q_file.name} after {elapsed:.1f}s: {e}")
            failed.append((q_file.name, str(e)))

    total_elapsed = time.time() - batch_t0
    avg = total_elapsed / max(1, n_to_run)
    print(f"\nFinished at {datetime.now().strftime('%H:%M:%S')}  "
          f"(total {total_elapsed:.1f}s, avg {avg:.1f}s/question)")
    print(f"Summary: {len(processed)} processed, {n_skipped} skipped, {len(failed)} failed")
    return {"processed": processed,
            "skipped":   [q.name for q in skipped_upfront],
            "failed":    failed}


## Check progress

Run this any time to see what's done and what's still pending. No work is performed — useful before kicking off `run_batch`, or after a partial run, or while planning which questions still need to go through.

In [ ]:
batch_status()


## Run the full batch

Each question prints `[i/N]` so you can see position in the run. To re-run a single question, delete its subfolder under `answers_langgraph_basic/` (or call `run_batch(overwrite=True)` for everything).

In [ ]:
results = await run_batch()
results
